# Notebook 04: Medical RAG (Retrieval-Augmented Generation) Pipeline
## Hallucination-Resistant Medical Report Generation Using Generative AI

### Research Objectives:
1. Ingest official radiology reference textbooks and clinical diagnostic criteria (PDF and TXT format).
2. Perform semantic text chunking with context overlap.
3. Generate dense vector representations using SentenceTransformers (`all-MiniLM-L6-v2`).
4. Build and query a FAISS vector database for relevant diagnostic context.
5. Enforce strict evidence boundary: Ensure retrieved general medical knowledge is never confused with patient-specific image findings.


In [ ]:
# 1. Module Imports and Configuration
import sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/hallucination_medical_ai')]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.config import get_config
from src.rag.document_loader import RadiologyDocumentLoader
from src.rag.chunking import SemanticChunker
from src.rag.vector_store import MedicalVectorStore
from src.rag.rag_pipeline import MedicalRAGPipeline

cfg = get_config()
print('Medical RAG pipeline initialized.')


### 2. Ingest and Chunk Radiology Knowledge Documents


In [ ]:
loader = RadiologyDocumentLoader(knowledge_dir=cfg.base_dir / 'data' / 'knowledge_base')
documents = loader.load_all_documents()
print(f'Loaded {len(documents)} source reference documents.')

chunker = SemanticChunker(chunk_size=256, chunk_overlap=32)
chunks = chunker.chunk_documents(documents)
print(f'Created {len(chunks)} semantic chunks.')
if chunks:
    print('Sample chunk preview:
', chunks[0]['content'][:200])


### 3. Build / Load FAISS Vector Index


In [ ]:
rag_pipeline = MedicalRAGPipeline(cfg)
rag_pipeline.build_or_load_index(chunks=chunks)
print(f'FAISS index active with {rag_pipeline.vector_store.total_vectors} vectors.')


### 4. Semantic Knowledge Retrieval Test
Query clinical diagnostic criteria based on detected visual findings.


In [ ]:
query = 'Radiological criteria for cardiomegaly and cardiac enlargement on PA chest X-ray'
retrieved_contexts = rag_pipeline.retrieve(query, top_k=3)
print(f'Top {len(retrieved_contexts)} retrieved clinical references:')
for i, ctx in enumerate(retrieved_contexts, 1):
    print(f'[{i}] Score: {ctx["score"]:.4f} | Source: {ctx["source"]} | Text: {ctx["text"][:150]}...')
